In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def mgd_tx60l(q):
    # Conversion en radians et calcul des cos/sin
    q = np.array(q, dtype=float)
    c = np.cos(q)
    s = np.sin(q)
    
    # T01 : translation  Z (0.375) + Rot Z (q1)
    T01 = np.array([[c[0], -s[0], 0, 0],
                    [s[0],  c[0], 0, 0],
                    [   0,     0, 1, 0.375],
                    [   0,     0, 0, 1]])

    # T12 : Rot X (90°) + Rot Z (q2)
    T12 = np.array([[c[1], -s[1], 0, 0],
                    [   0,     0,-1, 0],
                    [s[1],  c[1], 0, 0],
                    [   0,     0, 0, 1]])

    # T23 : translation X (0.02) + translation Y (0.400) + Rot Z (q3)
    T23 = np.array([[c[2], -s[2], 0, 0.020],
                    [s[2],  c[2], 0, 0.400],
                    [   0,     0, 1, 0],
                    [   0,     0, 0, 1]])

    # T34 : translation Y (0.450) + Rot X (-90°) + Rot Z (q4)
    T34 = np.array([[c[3], -s[3], 0, 0],
                    [   0,     0, 1, 0.450],
                    [-s[3],-c[3], 0, 0],
                    [   0,     0, 0, 1]])

    # T45 : Rot X (90°) + Rot Z (q5)
    T45 = np.array([[c[4], -s[4], 0, 0],
                    [   0,     0,-1, 0],
                    [s[4],  c[4], 0, 0],
                    [   0,     0, 0, 1]])

    # T56 : translation Y (0.070) + Rot X (-90°) + Rot Z (q6)
    T56 = np.array([[c[5], -s[5], 0, 0],
                    [   0,     0, 1, 0.070],
                    [-s[5],-c[5], 0, 0],
                    [   0,     0, 0, 1]])

    return T01 @ T12 @ T23 @ T34 @ T45 @ T56

T_resultat = mgd_tx60l([0, 0, 0, 0, 0, 0])
print(np.round(T_resultat, 4))

[[1.    0.    0.    0.02 ]
 [0.    1.    0.    0.   ]
 [0.    0.    1.    1.295]
 [0.    0.    0.    1.   ]]


In [6]:
#Verifie grossierement

#Verif sur axe 1
q = [np.pi/2, 0, 0, 0, 0, 0]
T_resultat = mgd_tx60l(q)
print("q1 =",q)
print(np.round(T_resultat, 4))

#Verif sur axe 2
q = [0, np.pi/2, 0, 0, 0, 0]
T_resultat = mgd_tx60l(q)
print("\nq2 =",q)
print(np.round(T_resultat, 4))

#Verif sur axe 3
q = [0, 0, np.pi/2, 0, 0, 0]
T_resultat = mgd_tx60l(q)
print("\nq3 =",q)
print(np.round(T_resultat, 4))

#Verif sur axe 4
q = [0, 0, 0, np.pi/2, 0, 0]
T_resultat = mgd_tx60l(q)
print("\nq4 =",q)
print(np.round(T_resultat, 4))

#Verif sur axe 5
q = [0, 0, 0, 0, np.pi/2, 0]
T_resultat = mgd_tx60l(q)
print("\nq5 =",q)
print(np.round(T_resultat, 4))

#Verif sur axe 6
q = [0, 0, 0, 0, 0, np.pi/2]
T_resultat = mgd_tx60l(q)
print("\nq6 =",q)
print(np.round(T_resultat, 4))

q1 = [1.5707963267948966, 0, 0, 0, 0, 0]
[[ 0.    -1.     0.     0.   ]
 [ 1.     0.     0.     0.02 ]
 [ 0.     0.     1.     1.295]
 [ 0.     0.     0.     1.   ]]

q2 = [0, 1.5707963267948966, 0, 0, 0, 0]
[[ 0.     0.    -1.    -0.92 ]
 [ 0.     1.     0.     0.   ]
 [ 1.     0.     0.     0.395]
 [ 0.     0.     0.     1.   ]]

q3 = [0, 0, 1.5707963267948966, 0, 0, 0]
[[ 0.     0.    -1.    -0.5  ]
 [ 0.     1.     0.     0.   ]
 [ 1.     0.     0.     0.775]
 [ 0.     0.     0.     1.   ]]

q4 = [0, 0, 0, 1.5707963267948966, 0, 0]
[[ 0.    -1.     0.     0.02 ]
 [ 1.     0.     0.     0.   ]
 [ 0.     0.     1.     1.295]
 [ 0.     0.     0.     1.   ]]

q5 = [0, 0, 0, 0, 1.5707963267948966, 0]
[[ 0.     0.    -1.    -0.05 ]
 [ 0.     1.     0.     0.   ]
 [ 1.     0.     0.     1.225]
 [ 0.     0.     0.     1.   ]]

q6 = [0, 0, 0, 0, 0, 1.5707963267948966]
[[ 0.    -1.     0.     0.02 ]
 [ 1.     0.     0.     0.   ]
 [ 0.     0.     1.     1.295]
 [ 0.     0.     0.     1.   ]]

In [4]:
def mgi_numerique(
    mgd_func, pos_cible, q_init, max_iter=100, tol=1e-5, damping=0.01
):
    """Calcul du MGI par algorithme itératif.

    :param mgd_func: Fonction mgd(q) renvoyant (pos, quat)
    :param pos_cible: Position visée [x, y, z]
    :param q_init: Configuration articulaire initiale (liste ou array)
    :return: (q_sol, succes)
    """
    q = np.array(q_init, dtype=float)
    target = np.array(pos_cible, dtype=float)

    for i in range(max_iter):
        pos_courante, _ = mgd_func(q)
        err = target - np.array(pos_courante)

        # Critère d'arrêt
        if np.linalg.norm(err) < tol:
            return q, True

        # Calcul numérique de la Jacobienne de position
        n = len(q)
        J = np.zeros((3, n))
        eps = 1e-6
        for j in range(n):
            q_eps = np.copy(q)
            q_eps[j] += eps
            pos_eps, _ = mgd_func(q_eps)
            J[:, j] = (np.array(pos_eps) - np.array(pos_courante)) / eps

        # Résolution par pseudo-inverse amortie
        J_t = J.T
        dq = J_t @ np.linalg.inv(J @ J_t + (damping**2) * np.eye(3)) @ err
        q += dq

    return q, False

# Wrapper pour extraire la position de la matrice 4x4
def mgd_pos_tx60l(q):
    T = mgd_tx60l(q)
    pos = T[:3, 3]  # Extraire [x, y, z] (colonne 3, lignes 0 à 2)
    return pos, T  # Renvoie un tuple (position, matrice_T)


# --- EXECUTION MGI ---

# 1. Cible et point de départ
target_pos = [0.6, 0.2, 0.3]
q_start = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

# 2. Calcul du MGI
q_sol, succes = mgi_numerique(mgd_pos_tx60l, target_pos, q_start)

# 3. Validation
if succes:
    pos_recalculee, _ = mgd_pos_tx60l(q_sol)
    erreur = np.linalg.norm(np.array(target_pos) - np.array(pos_recalculee))
    print(f"MGI Convergé : q = {np.round(q_sol, 4)}")
    print(f"Erreur de bouclage : {erreur:.6e} m")
else:
    print("Échec de convergence du MGI.")

MGI Convergé : q = [-15.3224 -43.1656  45.5498   7.6811   5.6589   0.    ]
Erreur de bouclage : 1.305340e-06 m


In [8]:
import time
import numpy as np
import pybullet as p

# 1. Récupération automatique de l'ID du robot chargé dans PyBullet
robot_id = p.getBodyUniqueId(0) # ou p.getBodyUniqueId(1) si un plan de sol (plane.urdf) est chargé en premier

# Affichage de la structure du robot pour vérifier les indices des articulations et du repère terminal
num_joints = p.getNumJoints(robot_id)
print(f"Robot ID : {robot_id} | Nombre de joints : {num_joints}")
for j in range(num_joints):
    info = p.getJointInfo(robot_id, j)
    print(f"Index {j} : joint = {info[1].decode('utf-8')} | lien = {info[12].decode('utf-8')}")

error: Not connected to physics server.

In [7]:
joint_indices = [0, 1, 2, 3, 4, 5] # À adapter si le joint 0 est une base fixe
link_index = num_joints - 1          # Dernier repère de la chaîne (ex: tool0)

N = 1000
q_samples = np.random.uniform(-np.pi/2, np.pi/2, size=(N, 6))

# --- 1. MGD analytique ---
t0 = time.perf_counter()
res_mgd = [mgd_tx60l(q) for q in q_samples]
t_mgd = (time.perf_counter() - t0) / N

# --- 2. MGD simulateur (PyBullet) ---
err_pos = []
err_rot = []

t0 = time.perf_counter()
for i, q in enumerate(q_samples):
    for j, j_idx in enumerate(joint_indices):
        p.resetJointState(robot_id, j_idx, q[j])
    state = p.getLinkState(robot_id, link_index, computeForwardKinematics=True)
    
    pos_sim = np.array(state[4])
    rot_sim = np.array(p.getMatrixFromQuaternion(state[5])).reshape(3, 3)
    
    T_mgd = res_mgd[i]
    err_pos.append(np.linalg.norm(T_mgd[:3, 3] - pos_sim))
    err_rot.append(np.linalg.norm(T_mgd[:3, :3] - rot_sim, ord='fro'))

t_sim = (time.perf_counter() - t0) / N

print(f"\nErreur position max : {np.max(err_pos):.3e} m | moyenne : {np.mean(err_pos):.3e} m")
print(f"Erreur rotation max : {np.max(err_rot):.3e}   | moyenne : {np.mean(err_rot):.3e}")
print(f"Temps CPU moyen MGD analytique : {t_mgd*1e6:.2f} µs")
print(f"Temps CPU moyen PyBullet       : {t_sim*1e6:.2f} µs (ratio : {t_sim/t_mgd:.1f}x)")

pybullet build time: Jan 29 2025 23:19:57


NameError: name 'robot_id' is not defined